# 🔧 Resolvendo o Problema de Corte Unidimensional com GRASP

**Objetivo:** Neste notebook, vamos resolver o Problema de Corte Unidimensional (Cutting Stock Problem) usando o algoritmo GRASP (Greedy Randomized Adaptive Search Procedure). Vamos explicar cada passo de forma detalhada para que mesmo quem está começando na área de otimização consiga entender.

---

## 📋 O que é o Problema de Corte Unidimensional?

Imagine que você tem barras de materiais (como vigas de concreto pré-moldado) com um tamanho fixo e precisa cortá-las em pedaços menores de tamanhos específicos para atender a um pedido. O desafio é:

1. **Atender toda a demanda** (produzir exatamente a quantidade necessária de cada tipo de peça)
2. **Minimizar o desperdício** (usar o menor número possível de barras)
3. **Não exceder o tamanho da barra** ao fazer os cortes

Este é um problema clássico de otimização combinatória que aparece em indústrias de corte de madeira, metal, papel, tecido, etc.

---

## 📊 Dados do Nosso Problema

Vamos começar carregando os dados que já temos disponíveis:

In [1]:
import pandas as pd

# Carregando os dados do Google Sheets
url = "https://docs.google.com/spreadsheets/d/17yC9PcTkC3gY5orbQ5VQXeBDOStMx2Dfpo53cMZTn5k/export?format=xlsx"

df_tipos_de_vigas = pd.read_excel(url, sheet_name="Tipos de vigas")
df_capacidade_das_formas = pd.read_excel(url, sheet_name="Capacidade das fôrmas")

print("📋 Tipos de vigas disponíveis:")
print(df_tipos_de_vigas)

print("\n📏 Capacidade das fôrmas (barras):")
print(df_capacidade_das_formas)

# Vamos também preparar os dados em formato mais fácil de usar
tipos_vigas = df_tipos_de_vigas.set_index('Tipo')['Tamanho (m)'].to_dict()
demanda = df_tipos_de_vigas.set_index('Tipo')['Demanda'].to_dict()
capacidades = df_capacidade_das_formas.set_index('Tipo de fôrma')['Capacidade (m)'].to_dict()

print("\n🔢 Dados preparados para o algoritmo:")
print(f"Tipos de vigas: {tipos_vigas}")
print(f"Demanda: {demanda}")
print(f"Capacidades das fôrmas: {list(capacidades.values())}")

📋 Tipos de vigas disponíveis:
   Tipo  Tamanho (m)  Demanda
0     1         1.22       24
1     2         1.45       60
2     3         2.35       56
3     4         2.50       72
4     5         2.65       16
5     6         2.95       17
6     7         3.30       12

📏 Capacidade das fôrmas (barras):
   Tipo de fôrma  Capacidade (m)
0              1           11.95
1              2           11.95
2              3           11.95
3              4           11.95
4              5           11.95
5              6           11.95
6              7            5.95

🔢 Dados preparados para o algoritmo:
Tipos de vigas: {1: 1.22, 2: 1.45, 3: 2.35, 4: 2.5, 5: 2.65, 6: 2.95, 7: 3.3}
Demanda: {1: 24, 2: 60, 3: 56, 4: 72, 5: 16, 6: 17, 7: 12}
Capacidades das fôrmas: [11.95, 11.95, 11.95, 11.95, 11.95, 11.95, 5.95]


## 🧮 Implementando o Algoritmo GRASP Passo a Passo

Agora vamos implementar o algoritmo GRASP para nosso problema. Vamos dividir em funções claras:

1. **Função de construção gulosa aleatória:** Cria uma solução inicial
2. **Função de busca local:** Melhora a solução atual
3. **Função principal GRASP:** Repete construção + busca local várias vezes
4. **Funções auxiliares:** Para calcular custos, imprimir resultados, etc.

## 🛠️ Funções Auxiliares

Vamos definir algumas funções auxiliares que vamos usar ao longo do algoritmo.